# Sentiment tagging — build training data for the controlled model

Uses the shared `tag_text_with_sentiment_prefixes` from `sentiment_utils.py`:
3-sentence chunks, tagged with [NEGATIVE]/[NEUTRAL]/[POSITIVE]. This must
match what `evaluate.ipynb` uses at inference time, or the controlled model
sees a tagging style it wasn't fine-tuned on.

In [ ]:
from datasets import load_dataset
from sentiment_utils import add_sentiment_prefixes, calculate_sentiment_preservation

Sanity check on two toy examples before running this over the full dataset.

In [ ]:
article = "The flight was delayed for six hours, and the airline staff was incredibly rude. Passengers were left stranded without food."
good_summary = "Stranded passengers faced six-hour flight delays and poor treatment from airline staff."
bad_summary = "Passengers enjoyed waiting at the airport despite a slight shift in flight schedules."

print("Good, sentiment-preserving summary:")
print(calculate_sentiment_preservation(article, good_summary))

print("\nBad summary that flips the tone:")
print(calculate_sentiment_preservation(article, bad_summary))

## Build the tagged train/validation sets

In [ ]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Small slices to fit Colab's T4 time/memory limits
train_subset = dataset['train'].select(range(2000))
val_subset = dataset['validation'].select(range(200))

print("Tagging training set...")
tagged_train_dataset = train_subset.map(add_sentiment_prefixes)

print("Tagging validation set...")
tagged_val_dataset = val_subset.map(add_sentiment_prefixes)

print("\nSample tagged article:")
print(tagged_train_dataset[0]['article_with_sentiment'][:500] + "...")

In [ ]:
tagged_train_dataset.save_to_disk("./data/tagged_train_nltk")
tagged_val_dataset.save_to_disk("./data/tagged_val_nltk")
print("Saved to ./data/tagged_train_nltk and ./data/tagged_val_nltk")